# Adding a pair of non-equal adjascent heat sources in 2D

This notebook demonstrates and validates the PhotonicsLAB project against the generation and analysis of a pair of nonequal, adjascent 2D Gaussian heat sources, their thermal lensing profiles and the corresponding propagation of arbitrary LG-mode seeds.

The subsequent notebooks will demonstrate how to add source profiles from an image file, how temporal and moving sources can be add to an arbitrary source profile, how a modulated seed can be used as input to a subsequent source profile, and lastly simulation of a pixelated gain laser for tunable, tailored generation of arbitrary beams.

<div class="alert alert-block alert-danger"> **Danger:** Not all the links below are correclty hyperlinked to their respective notebook. This is a TODO for when there's free time.</div>

# Table of Contents

1. [Preamble](#1.-Preamble)
2. [Generate heat sources](#2.-Generate-heat-sources)
3. [Solve 2D heat-equation using explicit finite-difference method](#3.-Solve-2D-heat-equation-using-explicit-finite-difference-method)
4. [Fit second-order polynomials to parabolic regions of temperature profile](#4.-Fit-second-order-polynomials-to-parabolic-regions-of-temperature-profile)
5. [Visualize and animate solution to the heat equation](#5.-Visualize-and-animate-solution-to-the-heat-equation)
6. [Phase modulated Gaussian beam generation and propagation](#6.-Phase-modulated-Gaussian-beam-generation-and-propagation)
7. [Phase modulated LG-mode generation and beam propagation](#7.-Phase-modulated-LG-mode-generation-and-beam-propagation)
8. [Adding sources from an image file](#8.-Adding-sources-from-an-image-file)
9. [Cardinal point diamond beam and donut focus dash dance](#9.-Cardinal-point-diamond-beam-and-donut-focus-dash-dance)
10. [Sequential self-thermal lensing using source from hdf5](#10.-Sequential-self-thermal-lensing-using-source-from-hdf5)
11. [49-Bit Pixelated Gain Laser shaping](#11.-49-Bit-Pixelated-Gain-Laser-Shaping)

## 1. Preamble
Please take note of the following [Jupyter Markdown Conventions](https://medium.com/ibm-data-science-experience/markdown-for-jupyter-notebooks-cheatsheet-386c05aeebed "Markdown for Jupyter Notebooks Cheatsheet") and make sure to refer to the [Jupiter Markdown Syntax Guide](https://sourceforge.net/p/jupiter/wiki/markdown_syntax/ "Markdon Syntx Guide") and the [Jupyter Notebook User's Manual](https://jupyter.brynmawr.edu/services/public/dblank/Jupyter%20Notebook%20Users%20Manual.ipynb "Jupyter Notebook User's Manual").

<div class="alert alert-block alert-success"> **Tip:** Ignoring these blocks is not likely to result in data or system functionality loss, however they do provide useful tips.

<div class="alert alert-block alert-info"> **Info:** These blocks contain useful and usually pertinent information.

<div class="alert alert-block alert-warning"> **Warning:** You should heed these blocks as they contain important information regarding implemetaion and / or usage. 

<div class="alert alert-block alert-danger"> **Danger:** Not taking the time to acknowledge / read these blocks coud results in loss of either data and / or system functionality! </div> </div> </div> </div>

<div class="alert alert-block alert-success"> **Tip:** Press `Ctrl-Shift-F` for the shortcut cheat-sheet.</div>

First we add the project's root path *(i.e. parent directory relative to the notebooks path)*, to the system's Python import paths, without this snippet Python would be unable to find and import the photonicslab package classes.


In [2]:
import os, sys
module_path = os.path.abspath(os.path.join('../'))
if module_path not in sys.path:
    sys.path.append(module_path)

Next we `import` the `photonicslab` package classes as well as `importlib`, which allows us to `reload` the relevent class should there have been any changes to it's corresponding modules.

<div class="alert alert-block alert-info"> **Info:** If you do not intend to make any changes directly to any of the `photonicslab` module source files, then it is not necessary to `import` and/or use the `importlib` package. Passing parameters to default constructors does not count as making changes to any of the `photonicslab` modules.</div>

In [3]:
import importlib
import photonicslab

/usr/lib/python3.6/site-packages/h5py/__init__.py:36: FutureWarning: Conversion of the second argument of issubdtype from `float` to `np.floating` is deprecated. In future, it will be treated as `np.float64 == np.dtype(float).type`.
  from ._conv import register_converters as _register_converters


## 2. Generate heat sources

- First we `reload` the `photonicslab.heateqn` module, in case there have been any changes made to it.
- Then we create a new `heateqn` object, called `mygaussQ` using the default constructor.
- Then we add a gaussian heat source to the center using the default constructor, *i.e. default method values*.
- Then we confirm that the initial heat source has been configured correctly by plotting it's output profile.
- Finally we retrieve the necessary parameters to solve the finite difference equation, using the `get_explicit_fd_params()` method.

<div class="alert alert-block alert-success"> **Tip:** Default constructor implies that no parameters are passed to method, i.e. we could adjust the coordinates and power of the gaussian heat source using `mygaussQ.add_gaussian_source(1.0, -0.5, q=50.)`.*</div>

<div class="alert alert-block alert-warning"> **Warning:** Need to verify physical parameters and units of Q, especially in relation to plots below.</div>

In [1]:
importlib.reload(photonicslab.heateqn)
mygaussQ = photonicslab.heateqn.HeatEqn2D(nt=200, nx=200, ny=200, hdf5filename='hdf5db/storage.hdf5', hdf5grp='nb1')
mygaussQ.add_gaussian(center_x=0.1, q=800)
mygaussQ.add_gaussian(center_x=-0.1)
mygaussQ.plot_source_profile(savepath='images/nb1_sourceq.png')
mygaussQ.get_explicit_fd_params()

NameError: name 'importlib' is not defined

<img width="800" height="600" align="left" src="images/nb1_sourceq.png">

<div class="alert alert-block alert-success"> **Tip:** Modern web-browers cache image and video files. It may be necessary to refresh the page with `F5`, to see the latest changes.</div>

## 3. Solve 2D heat-equation using explicit finite-difference method

- Again update any potential changes to the module using `importlib.reload()`.
- Here we create a new `fdsolver` object using the integration parameters retrieved in the previous step.
- The explicit finite difference solution is then stored in the hdf5 file, with indexing variable temperature, which can be adjusted through the parseable variable `hdf5table` when create a new instance of the `fdsolver` object.

In [5]:
importlib.reload(photonicslab.fdsolver)
myfdsolver = photonicslab.fdsolver.FDSolver2D(hdf5filename='hdf5db/storage.hdf5', hdf5grp='nb1')
myfdsolver.explicit_method()

## 4. Fit second-order polynomials to parabolic regions of temperature profile

 - Create a new `ThermalLens` class instance of the `phasemod` module.
 - Execute the `parafit` function on the newly create object, using non-default limits in y.
 - Lastly, run the `phase` function to determine the temporal phase profiles and store them in the `hdf5` file.

In [6]:
importlib.reload(photonicslab.phasemod)
myphasemod = photonicslab.phasemod.ThermalLens(hdf5filename='hdf5db/storage.hdf5', hdf5grp='nb1')
myphasemod.parabolic_fit(lim_y=[-0.1, 0.1])
myphasemod.phase()
myphasemod.parabolic_fit(hdf5table='phase')


<div class="alert alert-block alert-info"> **Info:** There's no intelligence in the steady state determination. Thumb-suck _big enough_ `nt`, and iterate through. Use the `default constructors` to adjust `nt`. Will need to develop convergence tests that can also accomodate three dimensions.</div>

## 5. Visualize and animate solution to the heat equation
<div class="alert alert-block alert-danger"> **Danger:** Do not commit `hdf5` databases into the Git repository. They consist of large filesizes!</div>

* We create `myview` and `myphaseview` visualization objects respectively and give them initial parameters:

* Generate and Plot steady state profiles.

* Generate and Play animation (loop) for transient profiles.

<div class="alert alert-block alert-warning"> **Warning:** Generation of `.MP4` files may take some time. Please be patient.</div>

In [17]:
importlib.reload(photonicslab.visualize)
myview = photonicslab.visualize.Visualize2D(hdf5filename='hdf5db/storage.hdf5', hdf5grp='nb1')
myview.steadystate_profiles(savepath='images/nb1_steady_state.png')
myview.transient_profiles(savepath='videos/nb1_transient.mp4')

myphaseview = photonicslab.visualize.VisualizePhase2D(hdf5filename='hdf5db/storage.hdf5', hdf5grp='nb1')
myphaseview.steadystate_profiles(savepath='images/nb1_steady_state_phase.png', threshold=.9)
myphaseview.transient_profiles(savepath='videos/nb1_transient_phase.mp4')

transient_profiles took 188.4376449584961 seconds.
_init_canvas took 0.17960453033447266 seconds.
steadystate_profiles took 1.7686707973480225 seconds.
_init_canvas took 0.1654338836669922 seconds.
transient_profiles took 190.67171168327332 seconds.


<img width="800" height="600" align="left" src="images/nb1_steady_state.png"/>

<video width="800" height="600" controls autoplay loop src="videos/nb1_transient.mp4"/>

<img width="800" height="600" align="left" src="images/nb1_steady_state_phase.png"/>

<video width="800" height="600" controls autoplay loop src="videos/nb1_transient_phase.mp4"/>